# Middlewares (Part 3) — Extending Agent State

In this notebook, we extend the agent state through various **middlewares**. The goal is to enable short-term memory **within a single invocation**.

## Key concepts

- **`AgentState`** – The data container that flows through the agent's execution graph. You can subclass it to track extra fields (e.g., token counts, call counts).
- **`@after_model` middleware** – A decorator that registers a function to run automatically after every model call. Used here to intercept model responses and accumulate statistics.
- **`Annotated[int, operator.add]`** – Tells LangGraph how to *combine* values across multiple steps: here, `operator.add` means values are **summed** (accumulated) rather than overwritten.
- **`TodoListMiddleware`** – A built-in middleware that lets the agent maintain an internal TODO list across its reasoning steps (short-term planning memory within one call).
- **`Runtime`** – Object passed into middleware functions that gives access to shared agent infrastructure.

In [ ]:
# Install required packages.
# langchain-openai provides the ChatOpenAI model integration.
!pip install -q langchain langchain-openai

In [ ]:
import operator  # Used for operator.add — tells LangGraph to sum values across agent steps

from google.colab import userdata  # Securely reads secrets stored in Colab's user data
from langchain.agents import AgentState, create_agent  # AgentState is the base state class; create_agent builds the agent
from langchain.agents.middleware import after_model  # Decorator to hook into post-model-call events
from langchain.agents.middleware import TodoListMiddleware  # Built-in middleware for agent task/TODO list tracking
from langchain.messages import HumanMessage  # Represents a message from the user
from langchain_core.messages import BaseMessage  # Base class for all message types (used in type hints)
from langchain_openai import ChatOpenAI  # OpenAI LLM integration
from langgraph.runtime import Runtime  # Provides access to runtime infrastructure inside middleware
from typing import Annotated, List  # Annotated lets us attach metadata to types (used with operator.add)

# Read the OpenAI API key from Colab secrets — never hard-code API keys in your notebooks!
openai_api_key = userdata.get('OPENAI_API_KEY')

# Helper function to pretty-print a list of messages (the conversation history).
def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()

In [ ]:
# --- Middleware 1: Track Token Usage ---

# We extend AgentState with four extra integer fields to accumulate token counts.
# Annotated[int, operator.add] means: when multiple steps produce values,
# LangGraph will ADD them together instead of keeping only the last value.
class TrackUsageState(AgentState):
    input_tokens: Annotated[int, operator.add]    # Total tokens sent to the model
    cached_tokens: Annotated[int, operator.add]   # Tokens served from the model's cache (cheaper)
    output_tokens: Annotated[int, operator.add]   # Total tokens the model generated in response
    reasoning_tokens: Annotated[int, operator.add]  # Tokens used for internal reasoning (e.g., o-series models)


# The @after_model decorator registers this function as a hook that fires
# automatically after EVERY model call during the agent's execution loop.
@after_model(state_schema=TrackUsageState)
def track_usage(state: TrackUsageState, runtime: Runtime):
    # Get the last message in the conversation — this is the model's fresh response.
    last_message = state["messages"][-1]
    if last_message.usage_metadata is not None:
        input_tokens = last_message.usage_metadata.get("input_tokens", 0)
        cached_tokens = last_message.usage_metadata.get("input_token_details", {}).get("cache_read", 0)
        output_tokens = last_message.usage_metadata.get("output_tokens", 0)
        reasoning_tokens = last_message.usage_metadata.get("output_token_details", {}).get("reasoning", 0)

        # Return a dict of state updates. LangGraph will add these values
        # into the running totals stored in TrackUsageState.
        return { "input_tokens": input_tokens, "cached_tokens": cached_tokens, "output_tokens": output_tokens, "reasoning_tokens": reasoning_tokens }

In [ ]:
# --- Middleware 2: Count Model Calls ---

# Another custom state extension — just one field to count how many times the model was called.
class CountModelCallsState(AgentState):
    model_calls: Annotated[int, operator.add]  # Each call contributes +1 to the running total


# This middleware fires after every model call and increments the counter by 1.
# Because of Annotated[int, operator.add], the value is summed across all agent steps.
@after_model(state_schema=CountModelCallsState)
def count_model_calls(state: CountModelCallsState, runtime: Runtime):
    return { "model_calls": 1 }  # Always return 1; LangGraph accumulates the total

In [ ]:
# Build the agent and attach all three middlewares.
# Middlewares are applied in the ORDER they appear in the list.
agent = create_agent(
    # The underlying LLM. reasoning_effort="low" makes it faster/cheaper for demos.
    model=ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key, reasoning_effort="low"),
    middleware=[
        track_usage,          # After each model call: accumulate token counts
        count_model_calls,    # After each model call: increment the call counter
        TodoListMiddleware()  # Gives the agent a built-in TODO list for multi-step planning
    ]
)

In [ ]:
# Run the agent with a complex mathematical task that requires multi-step planning.
# Because of TodoListMiddleware, the agent will maintain an internal TODO list
# and work through the items one by one in a single invocation.
prove_irrational = agent.invoke(
    input={
        "messages": [
            HumanMessage("Prove that the square roots of 2 and 3 are irrational. First plan your actions, prepare a list of TODO items and then start working them one by one. I want you to provide at least 2 independent proofs for both tasks.")
        ]
    }
)

In [ ]:
# Print the full conversation: every message exchanged between the user, the AI,
# and any tool calls that occurred during the agent's reasoning steps.
print_conversation(prove_irrational['messages'])

In [ ]:
# Print the token usage and call statistics collected by our custom middlewares.
# These values accumulated across all internal model calls during this single .invoke().
print(f"Model calls: {prove_irrational['model_calls']}")
print(f"Input tokens: {prove_irrational['input_tokens']}; Cache read: {prove_irrational['cached_tokens']}")
print(f"Output tokens: {prove_irrational['output_tokens']}; Reasoning: {prove_irrational['reasoning_tokens']}")